# 03 — Feature Engineering with validation

In [7]:
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT/'data'/'processed'
AUDIT = PROJECT_ROOT/'reports'/'data_quality'
FIGURES = PROJECT_ROOT/'reports'/'figures'/'final_eda'
HANDOFF = PROJECT_ROOT/'reports'/'modelling_handoff'
for d in (PROCESSED,AUDIT,FIGURES,HANDOFF): d.mkdir(parents=True,exist_ok=True)
print('Project:',PROJECT_ROOT)

import ast
# Pure-NumPy fallback for environments where Windows blocks scikit-learn's DLL.
class BallTree:
    def __init__(self, coords, metric='haversine'):
        if metric != 'haversine':
            raise ValueError("This fallback supports only the haversine metric.")
        self.coords = np.asarray(coords, dtype=float)
        self.cell = np.deg2rad(0.01)
        self.bins = {}

        lat_deg = np.rad2deg(self.coords[:, 0])
        lon_deg = np.rad2deg(self.coords[:, 1])
        lat_bins = np.floor((lat_deg + 90) / 0.01).astype(int)
        lon_bins = np.floor((lon_deg + 180) / 0.01).astype(int) % 36000

        for i, key in enumerate(zip(lat_bins, lon_bins)):
            self.bins.setdefault(key, []).append(i)

    def query_radius(self, coords, r, count_only=False):
        coords = np.asarray(coords, dtype=float)
        counts = np.zeros(len(coords), dtype=int)
        radius_deg = np.rad2deg(r)

        for i, (lat, lon) in enumerate(coords):
            lat_bin = int(np.floor((np.rad2deg(lat) + 90) / 0.01))
            lon_bin = int(np.floor((np.rad2deg(lon) + 180) / 0.01)) % 36000
            lat_span = int(np.ceil(radius_deg / 0.01)) + 1
            cos_lat = max(abs(np.cos(lat)), 1e-12)
            lon_span = min(18000, int(np.ceil(radius_deg / cos_lat / 0.01)) + 1)

            candidates = []
            for dy in range(-lat_span, lat_span + 1):
                for dx in range(-lon_span, lon_span + 1):
                    candidates.extend(self.bins.get(
                        (lat_bin + dy, (lon_bin + dx) % 36000), ()
                    ))

            if candidates:
                points = self.coords[candidates]
                dlat = points[:, 0] - lat
                dlon = points[:, 1] - lon
                a = (np.sin(dlat / 2) ** 2
                     + np.cos(lat) * np.cos(points[:, 0]) * np.sin(dlon / 2) ** 2)
                counts[i] = np.count_nonzero(2 * np.arcsin(np.sqrt(np.clip(a, 0, 1))) <= r)

        return counts if count_only else None
    
import duckdb
LISTINGS_PATH=PROCESSED/'listings_master.parquet'
REVIEWS_PATH=PROCESSED/'reviews_clean.parquet'
for p in [LISTINGS_PATH,REVIEWS_PATH]:
    if not p.is_file():raise FileNotFoundError(p)
listings=pd.read_parquet(LISTINGS_PATH)
for col in ['id','location','country','price','latitude','longitude','room_type']:
    if col not in listings:raise ValueError('Missing required source column: '+col)
features=listings.copy()
features['_row_id']=np.arange(len(features))
print('Loaded',len(features),'listings')

Project: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks
Loaded 756312 listings


## 1. Check source record keys before engineering

In [8]:
bad_key=features[['location','id']].isna().any(axis=1)
dup_key=features.duplicated(['location','id'],keep=False)
if bad_key.any() or dup_key.any():
    p=AUDIT/'feature_key_exceptions.csv'
    features.loc[bad_key|dup_key,[c for c in ['country','location','id','source_file'] if c in features]].to_csv(p,index=False)
    raise ValueError(f'Invalid keys: {bad_key.sum()} null and {dup_key.sum()} duplicate rows. Review {p}')
# Overlapping markets are tracked separately; strict within-market keys verified.
overlap=features.groupby('id')['location'].nunique().gt(1).sum()
print('Listing IDs found across multiple markets:',overlap,'(must assess geographic overlap)')

Listing IDs found across multiple markets: 0 (must assess geographic overlap)


## 2. Property, amenities and price representation

In [9]:
features['price_numeric']=pd.to_numeric(
 features['price'].astype('string').str.replace(r'[^0-9.\-]','',regex=True),errors='coerce')
for col in ['accommodates','bedrooms','beds','minimum_nights']:
    if col in features:features[col+'_clean']=pd.to_numeric(features[col],errors='coerce')
if 'bathrooms_text' in features:
    features['bathrooms_numeric']=pd.to_numeric(features.bathrooms_text.astype('string')
        .str.extract(r'(\d+(?:\.\d+)?)',expand=False),errors='coerce')
elif 'bathrooms' in features:features['bathrooms_numeric']=pd.to_numeric(features.bathrooms,errors='coerce')
else:features['bathrooms_numeric']=np.nan

def parse_amenities(x):
    if isinstance(x,(list,tuple)):return [str(k).lower() for k in x]
    if x is None or (isinstance(x,float) and np.isnan(x)):return []
    try:
        a=ast.literal_eval(str(x));return [str(k).lower() for k in a] if isinstance(a,(list,tuple)) else []
    except (ValueError,SyntaxError,TypeError):return []
if 'amenities' in features:
    amenities=features.amenities.map(parse_amenities)
    features['amenities_count']=amenities.map(len)
    for new,words in {'has_wifi':['wifi','wi-fi'],'has_kitchen':['kitchen'],
       'has_air_conditioning':['air conditioning'],'has_parking':['parking'],
       'has_pool':['pool'],'has_workspace':['dedicated workspace']}.items():
        features[new]=amenities.map(lambda a:int(any(w in v for v in a for w in words)))
    del amenities
if 'host_is_superhost' in features:
    features['is_superhost']=features.host_is_superhost.astype('string').str.lower().map(
        {'t':1,'f':0,'true':1,'false':0}).astype('Int8')
if 'availability_365' in features:
    a=pd.to_numeric(features.availability_365,errors='coerce')
    features['availability_rate_365']=a.where(a.between(0,365))/365

## 3. Spatial competition features
Uses all listings with valid coordinates, including listings without a price. Counts same-market competitors; duplicates across overlapping market files remain a documented caveat.

In [10]:
R=6371.0088
for col in ['latitude','longitude']:
    features[col]=pd.to_numeric(features[col],errors='coerce')
geo=features.latitude.between(-90,90)&features.longitude.between(-180,180)&~((features.latitude==0)&(features.longitude==0))&features.location.notna()
radii={'competitors_500m':.5,'competitors_1km':1.,'competitors_2km':2.}
for c in list(radii)+['comparable_competitors_1km']:features[c]=np.nan
for loc,group in features.loc[geo].groupby('location'):
    coords=np.radians(group[['latitude','longitude']].to_numpy(dtype=float))
    tree=BallTree(coords,metric='haversine')
    for col,r in radii.items():
        features.loc[group.index,col]=tree.query_radius(coords,r=r/R,count_only=True)-1
    print('Competition calculated:',loc,len(group))
for (loc,room),group in features.loc[geo&features.room_type.notna()].groupby(['location','room_type']):
    coords=np.radians(group[['latitude','longitude']].to_numpy(dtype=float))
    tree=BallTree(coords,metric='haversine')
    features.loc[group.index,'comparable_competitors_1km']=tree.query_radius(coords,r=1./R,count_only=True)-1

Competition calculated: Amsterdam 10369
Competition calculated: Antwerp 2535
Competition calculated: Athens 14337
Competition calculated: Barcelona 15293
Competition calculated: Bergamo 4237
Competition calculated: Berlin 12776
Competition calculated: Bologna 4948
Competition calculated: Bordeaux 12217
Competition calculated: Bristol 3023
Competition calculated: Brussels 5078
Competition calculated: Copenhagen 23144
Competition calculated: Crete 27333
Competition calculated: Dublin 6966
Competition calculated: Edinburgh 6244
Competition calculated: Euskadi 6248
Competition calculated: Florence 13472
Competition calculated: Geneva 2593
Competition calculated: Ghent 1383
Competition calculated: Girona 21257
Competition calculated: Greater Manchester 6930
Competition calculated: Lisbon 24876
Competition calculated: London 92638
Competition calculated: Lyon 9317
Competition calculated: Madrid 22708
Competition calculated: Mallorca 14817
Competition calculated: Menorca 3620
Competition calc

## 4. Historical review-activity features

In [11]:
con=duckdb.connect()
con.execute("SET memory_limit='12GB'");con.execute('SET threads=4')
q=str(REVIEWS_PATH).replace("'","''")
con.execute(f"CREATE VIEW reviews AS SELECT location, CAST(listing_id AS VARCHAR) AS listing_key, TRY_CAST(date AS DATE) AS d FROM read_parquet('{q}')")
cutoff=con.execute('SELECT MAX(d) FROM reviews').fetchone()[0]
if cutoff is None:raise ValueError('No valid review dates')
print('Review cutoff:',cutoff)
review_summary=con.execute('''SELECT location,listing_key,
 COUNT(*) FILTER(WHERE d >= ?::DATE - INTERVAL 29 DAYS AND d <= ?::DATE) AS reviews_last_30d,
 COUNT(*) FILTER(WHERE d >= ?::DATE - INTERVAL 89 DAYS AND d <= ?::DATE) AS reviews_last_90d,
 COUNT(*) FILTER(WHERE d >= ?::DATE - INTERVAL 364 DAYS AND d <= ?::DATE) AS reviews_last_365d,
 MAX(d) FILTER (WHERE d <= ?::DATE) AS last_review_date
 FROM reviews WHERE listing_key IS NOT NULL AND d <= ?::DATE
 GROUP BY location,listing_key''',[cutoff]*8).df()
con.close()
review_summary['last_review_date']=pd.to_datetime(review_summary.last_review_date)
review_summary['days_since_last_review']=(pd.Timestamp(cutoff)-review_summary.last_review_date).dt.days
print('Review summary rows:',len(review_summary))

Review cutoff: 2026-08-11
Review summary rows: 609620


In [12]:
# Join by market + listing key. Preserve original ID values and prevent row explosion.
features['_listing_key']=features.id.astype('string').str.strip()
review_summary['listing_key']=review_summary.listing_key.astype('string').str.strip()
assert not review_summary.duplicated(['location','listing_key']).any()
features=features.merge(review_summary.drop(columns='last_review_date'),
 left_on=['location','_listing_key'],right_on=['location','listing_key'],
 how='left',validate='many_to_one')
for col in ['reviews_last_30d','reviews_last_90d','reviews_last_365d']:
    features[col]=features[col].fillna(0).astype('int32')
features=features.drop(columns=['_listing_key','listing_key'])
assert len(features)==len(listings)

## 5. Neighborhood and pricing benchmarks


In [13]:
hood=next((c for c in ['neighbourhood_cleansed','neighbourhood'] if c in features),None)
if hood is None:
    features['neighbourhood_cleansed']=pd.NA;hood='neighbourhood_cleansed'
hood_stats=(features.dropna(subset=[hood]).groupby(['country','location',hood],dropna=False)
 .agg(neighbourhood_listing_count=('id','size'),neighbourhood_host_count=(
    'host_id','nunique') if 'host_id' in features else ('id','nunique'))
 .reset_index())
features=features.merge(hood_stats,on=['country','location',hood],how='left',validate='many_to_one')
features['location_listing_count']=features.groupby('location')['id'].transform('size')
features['market_median_price']=features.groupby('location').price_numeric.transform('median')
features['neighbourhood_median_price']=features.groupby(['location',hood]).price_numeric.transform('median')
p=features.price_numeric;m=features.neighbourhood_median_price
features['price_vs_neighbourhood_pct']=((p/m)-1).where(p.gt(0)&m.gt(0))*100

## 6. Final data-quality checks — integrated into feature engineering

In [14]:
def gate(name,passed,detail=''):
    return {'gate':name,'passed':bool(passed),'detail':detail}
gates=[gate('row_count_preserved',len(features)==len(listings)),
 gate('source_keys_unique',not features.duplicated(['location','id']).any()),
 gate('row_ids_unique',features['_row_id'].is_unique),
 gate('review_count_monotone',(features.reviews_last_30d.le(features.reviews_last_90d)&
   features.reviews_last_90d.le(features.reviews_last_365d)).all())]
geo_cols=['competitors_500m','competitors_1km','competitors_2km']
v=features[geo_cols].dropna()
gates.append(gate('competition_monotone',((v[geo_cols[0]]<=v[geo_cols[1]])&
 (v[geo_cols[1]]<=v[geo_cols[2]])&(v>=0).all(axis=1)).all()))
if 'availability_rate_365' in features:
    gates.append(gate('availability_0_to_1',features.availability_rate_365.dropna().between(0,1).all()))
# Scientific validity warnings are NOT magically cleared by code execution.
gates.append(gate('cross_country_price_currency_verified',False,'Not independently source-verified / normalized'))
gates.append(gate('swiss_price_scale_resolved',False,'Geneva/Vaud scale 0–0.99 requires source evidence'))
gates_df=pd.DataFrame(gates)
display(gates_df);gates_df.to_csv(AUDIT/'quality_gates.csv',index=False)
if not gates_df.loc[~gates_df.gate.isin(['cross_country_price_currency_verified','swiss_price_scale_resolved']),'passed'].all():
    raise ValueError('Structural feature validation failed. Review reports/data_quality/quality_gates.csv')
feature_missing=pd.DataFrame({'feature':features.columns,'missing_pct':(features.isna().mean()*100).round(2).values})
feature_missing.to_csv(AUDIT/'engineered_feature_missingness.csv',index=False)

,gate,passed,detail
0,row_count_preserved,True,
1,source_keys_unique,True,
2,row_ids_unique,True,
3,review_count_monotone,True,
4,competition_monotone,True,
5,availability_0_to_1,True,
6,cross_country_price_currency_verified,False,Not independently source-verified / normalized
7,swiss_price_scale_resolved,False,Geneva/Vaud scale 0–0.99 requires source evidence


In [16]:
# Exclude Switzerland from price-based modelling
# while retaining the listings for non-price analysis.

features["exclude_from_price_modelling"] = (
    features["country"].eq("Switzerland")
)

features["exclusion_reason"] = np.where(
    features["exclude_from_price_modelling"],
    "Unresolved price scaling in Geneva/Vaud",
    pd.NA
)

print("Swiss listings flagged:",
      features["exclude_from_price_modelling"].sum())

Swiss listings flagged: 7888


## 7. Persist feature dataset

In [17]:
out=PROCESSED/'listings_features.parquet'
features.drop(columns=['_row_id']).to_parquet(out,index=False,compression='zstd')
print('Saved',out,'rows',len(features),'columns',len(features.columns))

Saved C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\data\processed\listings_features.parquet rows 756312 columns 125
